<a href="https://colab.research.google.com/github/Makayla-Kelly/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07: Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Makayla-Kelly/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring. This is a transparent, March-only decision-support rule. It is not a model and does not claim to predict Google's algorithm.

## 1. My rule, signal checks, and reason code

**Rule in plain language:** Prioritize a page for a human CTR review when it was last updated at least 180 days before the 2026-03-31 decision point, still has meaningful March search visibility, ranks in positions 4–20, and its March CTR is below the March median for comparable positions. Higher visibility and a larger below-norm CTR gap rank first.

**Signal check 1: CTR versus position (flag-linked):** CTR-fix logic is only reasonable if CTR differs systematically by search position. The bucket table below reports the observed March median CTR and n for each position range.

**Signal check 2: staleness plus visibility (refresh-linked):** A refresh-review rule is only useful if some older pages remain visible enough to merit review. The bucket table below reports n, March median impressions, and the share with at least 500 March impressions.

**Reason code:** stale_visible_ctr_gap  
**Action label:** REVIEW_CTR_OPPORTUNITY

The signal verdicts are calculated from March observations only. April outcomes, labels, product flags, client names, URLs, and raw queries are never inputs.

In [1]:
!pip -q install -U duckdb huggingface_hub

from pathlib import Path
import json
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata
from IPython.display import display

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add your HF_TOKEN in Colab Secrets, then run this notebook again.')

con = duckdb.connect()
con.execute('SET enable_progress_bar = false')
con.execute('SET VARIABLE hf_token = ?', [HF_TOKEN])
con.execute('''
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
''')

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"{REL}/fact_content_daily_performance"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
DECISION_DATE = '2026-03-31'

# This frame contains only measurements available by the March 31 decision point.
baseline = con.sql(f'''
WITH march_content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_31d,
        SUM(gsc_clicks) AS clicks_31d,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_31d,
        SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_31d,
        COUNT(*) AS gsc_days_observed
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
)
SELECT
    m.*,
    DATEDIFF('day', CAST(d.content_updated_date AS DATE), DATE '{DECISION_DATE}') AS days_since_update
FROM march_content AS m
INNER JOIN {DIM} AS d
    USING (client_hash_id, content_hash_id)
WHERE d.content_updated_date IS NOT NULL
  AND CAST(d.content_updated_date AS DATE) <= DATE '{DECISION_DATE}'
''').df()

# Exclude incomplete March histories rather than interpreting missing GSC days as zero activity.
baseline = baseline.loc[baseline['gsc_days_observed'] >= 25].copy()
baseline['position_bucket'] = pd.cut(
    baseline['avg_position_31d'],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=['1–3', '4–10', '11–20', '21–50', '51+'],
    include_lowest=True,
)
baseline['staleness_bucket'] = pd.cut(
    baseline['days_since_update'],
    bins=[-1, 89, 179, 364, np.inf],
    labels=['0–89 days', '90–179 days', '180–364 days', '365+ days'],
)
baseline['position_norm_ctr'] = baseline.groupby('position_bucket', observed=False)['ctr_31d'].transform('median')
baseline['ctr_gap'] = baseline['position_norm_ctr'] - baseline['ctr_31d']
baseline['relative_ctr_deficit'] = (baseline['ctr_gap'] / baseline['position_norm_ctr']).clip(lower=0).fillna(0)

# Signal check 1: CTR should differ by position before a CTR-versus-position rule is trusted.
position_check = (
    baseline.groupby('position_bucket', observed=False)
    .agg(
        n=('content_hash_id', 'size'),
        median_ctr=('ctr_31d', 'median'),
        median_impressions=('impressions_31d', 'median'),
    )
    .reset_index()
)
top_three_ctrs = position_check.loc[position_check['position_bucket'].isin(['1–3', '4–10', '11–20']), 'median_ctr'].to_numpy()
position_verdict = 'CONFIRMED' if len(top_three_ctrs) == 3 and np.all(np.diff(top_three_ctrs) < 0) else 'MIXED'

# Signal check 2: older content must still have material search visibility to support review.
baseline['visible_500'] = baseline['impressions_31d'] >= 500
staleness_check = (
    baseline.groupby('staleness_bucket', observed=False)
    .agg(
        n=('content_hash_id', 'size'),
        median_impressions=('impressions_31d', 'median'),
        visible_500_share=('visible_500', 'mean'),
    )
    .reset_index()
)
older_visible_share = baseline.loc[baseline['days_since_update'] >= 180, 'visible_500'].mean()
staleness_verdict = 'CONFIRMED' if pd.notna(older_visible_share) and older_visible_share >= 0.25 else 'MIXED'

print('Signal check 1 — CTR versus position (flag-linked)')
display(position_check.assign(median_ctr=lambda df: df['median_ctr'].round(4)))
print(f'Verdict: {position_verdict}')

print('\nSignal check 2 — staleness plus March visibility (refresh-linked)')
display(staleness_check.assign(visible_500_share=lambda df: df['visible_500_share'].round(3)))
print(f'Verdict: {staleness_verdict}')
print(f'\nEligible March-only content rows after the 25-day guard: {len(baseline):,}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 19.5 MB/s eta 0:00:00
Signal check 1 — CTR versus position (flag-linked)


,position_bucket,n,median_ctr,median_impressions
0,1–3,2199,0.0015,1246.0
1,4–10,7690,0.0011,685.0
2,11–20,3600,0.0000,511.5
3,21–50,4306,0.0000,810.5
4,51+,532,0.0000,207.0


Verdict: CONFIRMED

Signal check 2 — staleness plus March visibility (refresh-linked)


,staleness_bucket,n,median_impressions,visible_500_share
0,0–89 days,18185,682.0,0.592
1,90–179 days,127,276.0,0.417
2,180–364 days,15,290.0,0.333
3,365+ days,0,NaN,NaN


Verdict: CONFIRMED

Eligible March-only content rows after the 25-day guard: 18,327


## 2. Build the ranked queue (writes the CSV)

The score gives every eligible candidate a transparent priority based on March impressions and how far its CTR falls below the March median for its position bucket. It is a directional review priority, not a predicted business outcome.

In [2]:
MIN_DAYS_STALE = 180
MIN_MARCH_IMPRESSIONS = 500

baseline['eligible_for_review'] = (
    (baseline['days_since_update'] >= MIN_DAYS_STALE)
    & (baseline['impressions_31d'] >= MIN_MARCH_IMPRESSIONS)
    & baseline['avg_position_31d'].between(4, 20, inclusive='both')
    & (baseline['ctr_gap'] > 0)
)

# No learned weights: visible reach and the observed within-bucket CTR deficit drive priority.
baseline['baseline_action_score'] = np.where(
    baseline['eligible_for_review'],
    np.log1p(baseline['impressions_31d']) * (1 + baseline['relative_ctr_deficit']),
    0.0,
)
baseline['reason_code'] = np.where(
    baseline['eligible_for_review'],
    'stale_visible_ctr_gap',
    'not_ranked',
)
baseline['action_label'] = np.where(
    baseline['eligible_for_review'],
    'REVIEW_CTR_OPPORTUNITY',
    'NO_ACTION',
)

queue_columns = [
    'client_hash_id', 'content_hash_id', 'action_label', 'reason_code',
    'baseline_action_score', 'impressions_31d', 'clicks_31d', 'ctr_31d',
    'avg_position_31d', 'position_norm_ctr', 'ctr_gap', 'relative_ctr_deficit',
    'days_since_update', 'gsc_days_observed',
]
queue = (
    baseline.loc[baseline['eligible_for_review'], queue_columns]
    .sort_values(['baseline_action_score', 'impressions_31d'], ascending=False)
    .reset_index(drop=True)
)
queue.insert(0, 'rank', np.arange(1, len(queue) + 1))

OUTPUT_DIR = Path('work/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIR / 'baseline_action_score.csv'
metrics_path = OUTPUT_DIR / 'baseline_action_score_metrics.json'
queue.to_csv(csv_path, index=False)

metrics = {
    'decision_date': DECISION_DATE,
    'source_window': 'March 2026 only',
    'rule': 'stale (180+ days), visible (500+ March impressions), position 4–20, and below-position-norm CTR',
    'reason_code': 'stale_visible_ctr_gap',
    'action_label': 'REVIEW_CTR_OPPORTUNITY',
    'eligible_rows': int(len(queue)),
    'signal_verdicts': {
        'ctr_vs_position': position_verdict,
        'staleness_plus_visibility': staleness_verdict,
    },
}
metrics_path.write_text(json.dumps(metrics, indent=2))

print(f'Ranked queue rows: {len(queue):,}')
print(f'Wrote CSV: {csv_path}')
print(f'Wrote metrics receipt: {metrics_path}')
display(queue.head(10).round({'baseline_action_score': 3, 'ctr_31d': 4, 'position_norm_ctr': 4, 'ctr_gap': 4}))


Ranked queue rows: 1
Wrote CSV: work/outputs/baseline_action_score.csv
Wrote metrics receipt: work/outputs/baseline_action_score_metrics.json


,rank,client_hash_id,content_hash_id,action_label,reason_code,baseline_action_score,impressions_31d,clicks_31d,ctr_31d,avg_position_31d,position_norm_ctr,ctr_gap,relative_ctr_deficit,days_since_update,gsc_days_observed
0,1,client_c182d11e4862a37d,content_bea86ce3455100b0,REVIEW_CTR_OPPORTUNITY,stale_visible_ctr_gap,14.375,3670.0,1.0,0.0003,6.517166,0.0011,0.0008,0.751285,232,31


## 3. Top-10 review

Each review is a decision-support note, not an automated action. The notebook writes one line for each of the ten highest-scoring candidates after the real March-only queue is built.

In [6]:
STALE_BONUS_DAYS = 180
MIN_MARCH_IMPRESSIONS = 500

baseline["eligible_for_review"] = (
    (baseline["impressions_31d"] >= MIN_MARCH_IMPRESSIONS)
    & baseline["avg_position_31d"].between(4, 20, inclusive="both")
    & (baseline["ctr_gap"] > 0)
)

baseline["staleness_bonus"] = (
    baseline["days_since_update"] >= STALE_BONUS_DAYS
).astype(int)

baseline["baseline_action_score"] = np.where(
    baseline["eligible_for_review"],
    np.log1p(baseline["impressions_31d"])
    * (
        1
        + baseline["relative_ctr_deficit"]
        + 0.25 * baseline["staleness_bonus"]
    ),
    0.0,
)

baseline["reason_code"] = np.where(
    baseline["eligible_for_review"],
    "visible_ctr_gap",
    "not_ranked",
)

baseline["action_label"] = np.where(
    baseline["eligible_for_review"],
    "REVIEW_CTR_OPPORTUNITY",
    "NO_ACTION",
)

queue_columns = [
    "client_hash_id",
    "content_hash_id",
    "action_label",
    "reason_code",
    "baseline_action_score",
    "impressions_31d",
    "clicks_31d",
    "ctr_31d",
    "avg_position_31d",
    "position_norm_ctr",
    "ctr_gap",
    "relative_ctr_deficit",
    "days_since_update",
    "staleness_bonus",
    "gsc_days_observed",
]

queue = (
    baseline.loc[baseline["eligible_for_review"], queue_columns]
    .sort_values(
        ["baseline_action_score", "impressions_31d"],
        ascending=False,
    )
    .reset_index(drop=True)
)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "baseline_action_score.csv"
metrics_path = OUTPUT_DIR / "baseline_action_score_metrics.json"

queue.to_csv(csv_path, index=False)

metrics = {
    "decision_date": DECISION_DATE,
    "source_window": "March 2026 only",
    "rule": (
        "visible (500+ March impressions), position 4–20, "
        "and below-position-norm CTR; 180+ days since update "
        "is a ranking bonus"
    ),
    "reason_code": "visible_ctr_gap",
    "action_label": "REVIEW_CTR_OPPORTUNITY",
    "eligible_rows": int(len(queue)),
    "signal_verdicts": {
        "ctr_vs_position": position_verdict,
        "staleness_plus_visibility": staleness_verdict,
    },
}

metrics_path.write_text(json.dumps(metrics, indent=2))

print(f"Ranked queue rows: {len(queue):,}")
print(f"Wrote CSV: {csv_path}")
print(f"Wrote metrics receipt: {metrics_path}")

display(
    queue.head(10).round(
        {
            "baseline_action_score": 3,
            "ctr_31d": 4,
            "position_norm_ctr": 4,
            "ctr_gap": 4,
        }
    )
)

Ranked queue rows: 1,440
Wrote CSV: work/outputs/baseline_action_score.csv
Wrote metrics receipt: work/outputs/baseline_action_score_metrics.json


,rank,client_hash_id,content_hash_id,action_label,reason_code,baseline_action_score,impressions_31d,clicks_31d,ctr_31d,avg_position_31d,position_norm_ctr,ctr_gap,relative_ctr_deficit,days_since_update,staleness_bonus,gsc_days_observed
0,1,client_3197e6291363b4db,content_22588e765b93dfac,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,20.139,38813.0,4.0,0.0001,7.626955,0.0011,0.0010,0.905930,34,0,31
1,2,client_3197e6291363b4db,content_93d76695da196fdf,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,19.735,19292.0,0.0,0.0000,8.243572,0.0011,0.0011,1.000000,34,0,31
2,3,client_73cda7b4e4f265ea,content_345c8feeab5d080a,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,19.730,23418.0,1.0,0.0000,4.969980,0.0011,0.0011,0.961022,34,0,31
3,4,client_3197e6291363b4db,content_33c141e04352003a,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,17.642,13108.0,2.0,0.0002,8.489854,0.0011,0.0009,0.860728,34,0,31
4,5,client_73cda7b4e4f265ea,content_e080fc5170d28a55,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,17.594,12913.0,2.0,0.0002,5.328971,0.0011,0.0009,0.858625,34,0,31
5,6,client_cd12bcfd98942aa1,content_99fc6465edb0e52c,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,17.446,6143.0,0.0,0.0000,9.945141,0.0011,0.0011,1.000000,34,0,31
6,7,client_73cda7b4e4f265ea,content_034f3d6a57be0366,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,17.368,14554.0,3.0,0.0002,4.509963,0.0011,0.0009,0.811848,34,0,31
7,8,client_73cda7b4e4f265ea,content_94b6237a841e9354,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,17.186,13848.0,3.0,0.0002,4.071563,0.0011,0.0009,0.802256,34,0,31
8,9,client_73cda7b4e4f265ea,content_c0bfca959627c226,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,17.168,16065.0,4.0,0.0002,4.475195,0.0011,0.0008,0.772727,34,0,31
9,10,client_73cda7b4e4f265ea,content_ed9c7fe5a778c796,REVIEW_CTR_OPPORTUNITY,visible_ctr_gap,17.099,5162.0,0.0,0.0000,4.489151,0.0011,0.0011,1.000000,34,0,31


## 4. Weak picks + leakage check

The weakest candidates among the top ten deserve the first skeptical review. This rule uses no April performance, labels, FlyRank decision flags, client names, URLs, or query text.

In [4]:
weak_picks = (
    top_10.sort_values(
        ['relative_ctr_deficit', 'gsc_days_observed', 'impressions_31d'],
        ascending=[True, True, True],
    )
    .loc[:, ['rank', 'content_hash_id', 'baseline_action_score', 'impressions_31d', 'ctr_31d', 'position_norm_ctr', 'ctr_gap', 'days_since_update', 'gsc_days_observed']]
    .head(3)
)

print('Weakest top-10 picks to review first:')
display(weak_picks.round({'baseline_action_score': 3, 'ctr_31d': 4, 'position_norm_ctr': 4, 'ctr_gap': 4}))
print(
    'These are weaker because a small observed CTR gap, limited observation history, or lower reach can make '
    'the rule sensitive to normal query-mix variation. A human should inspect intent, the current search-result page, '
    'and any changes not represented in the warehouse before acting.'
)

rule_input_columns = [
    'impressions_31d', 'ctr_31d', 'avg_position_31d', 'days_since_update',
    'gsc_days_observed', 'position_norm_ctr', 'ctr_gap',
]
forbidden_terms = ('april', 'future', 'label', 'outcome', 'flag', 'priority_score', 'action_type')
leaky_inputs = [
    column for column in rule_input_columns
    if any(term in column.lower() for term in forbidden_terms)
]
assert not leaky_inputs, f'Leakage check failed: {leaky_inputs}'
print('\nLeakage check: PASS — the baseline score uses March-only observed or March-derived inputs.')


Weakest top-10 picks to review first:


,rank,content_hash_id,baseline_action_score,impressions_31d,ctr_31d,position_norm_ctr,ctr_gap,days_since_update,gsc_days_observed
0,1,content_bea86ce3455100b0,14.375,3670.0,0.0003,0.0011,0.0008,232,31


These are weaker because a small observed CTR gap, limited observation history, or lower reach can make the rule sensitive to normal query-mix variation. A human should inspect intent, the current search-result page, and any changes not represented in the warehouse before acting.

Leakage check: PASS — the baseline score uses March-only observed or March-derived inputs.
